In [ ]:
import pymysql

tables = []
def get_all_table_ddls(host, port, user, password, database, charset='utf8'):
    global tables
    """
    连接到MySQL数据库并获取指定数据库下所有表的DDL。

    Args:
        host (str): 数据库主机名或IP地址。
        port (int): 数据库端口号。
        user (str): 数据库用户名。
        password (str): 数据库密码。
        database (str): 要获取DDL的数据库名称。
        charset (str): 连接字符集，默认为'utf8'。

    Returns:
        dict: 一个字典，键是表名，值是对应的CREATE TABLE语句。
              如果连接失败或没有表，则返回空字典。
    """
    conn = None
    cursor = None
    table_ddls = {}
    try:
        # 建立数据库连接
        conn = pymysql.connect(
            host=host,
            port=port,
            user=user,
            password=password,
            database=database,
            charset=charset,
            # useSSL=false 对应 ssl=False
            ssl=False
            # serverTimezone=GMT%2B8 和 characterEncoding=utf-8 已经在 charset 中处理
        )
        cursor = conn.cursor()

        # 获取所有表名
        cursor.execute(f"SHOW TABLES IN `{database}`")
        tables = [row[0] for row in cursor.fetchall()]

        # 遍历表名，获取每个表的DDL
        for table_name in tables:
            # SHOW CREATE TABLE 返回两列：表名和 CREATE TABLE 语句
            cursor.execute(f"SHOW CREATE TABLE `{table_name}`")
            result = cursor.fetchone()
            if result and len(result) > 1:
                table_ddls[table_name] = result[1] # 第二列是 CREATE TABLE 语句

    except pymysql.Error as e:
        print(f"数据库连接或查询错误: {e}")
        # 可以选择重新抛出异常或返回空字典
        # raise e
    finally:
        # 确保关闭游标和连接
        if cursor:
            cursor.close()
        if conn:
            conn.close()

    return table_ddls

# 示例用法 (使用注释中的连接信息)
db_host = "mysql-8.summerfarm.net"
db_port = 3307
db_user = "dev"
db_password = "xianmu619"
db_name = "cosfodb"

all_ddls = get_all_table_ddls(db_host, db_port, db_user, db_password, db_name)

# # 打印结果
if all_ddls:
    for table, ddl in all_ddls.items():
        print(f"--- DDL for table: {table} ---")
        print(ddl)
        print("-" * 20)
else:
    print("未能获取到任何表的DDL。")
